In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

In [1]:
# ================================================================
# Imports
# ================================================================
import numpy as np
from tqdm import tqdm
from scipy.sparse import csr_matrix, coo_matrix, bmat, diags
from scipy.sparse.csgraph import connected_components
from scipy.spatial.distance import pdist, squareform
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr

# ================================================================
# Load data
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

# ================================================================
# Keep TWO largest connected components
# ================================================================
DIST_TH = 0.22

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
Dmat = squareform(pdist(xy))

W = (Dmat <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords[use_idx]
y = y[use_idx]
W = W[use_idx][:, use_idx]

S, TT = y.shape
period = 52

print("Using S =", S)

# ================================================================
# CAR precision
# ================================================================
D = csr_matrix(np.diag(np.array(W.sum(axis=1)).flatten()))
prec = D - W

# ================================================================
# Global scaled trend
# ================================================================
t_full = np.arange(1, TT + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std()

# ================================================================
# Common MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

a_tau = 2
b_tau = 25

# ================================================================
# -------------------- p01 --------------------
# ================================================================
loc = np.where(y[:, :-1] == 0)
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:,0], pairs[:,1]))]
pairs[:,1] += 1

row_idx = pairs[:,0]
time_idx = pairs[:,1] - 1
N = len(row_idx)

next_y = y[pairs[:,0], pairs[:,1]]
kappa = next_y - 0.5

t = t_scaled[time_idx]

covariates = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*(time_idx+1)/period), np.cos(2*np.pi*(time_idx+1)/period),
    np.sin(2*np.pi*(time_idx+1)/period), np.sin(2*np.pi*(time_idx+1)/period),
    t, t
])

K = covariates.shape[1]
theta_dim = K*S

rows, cols, vals = [], [], []
for i in range(N):
    s = row_idx[i]
    for k in range(K):
        rows.append(i)
        cols.append(s + k*S)
        vals.append(covariates[i,k])

X = coo_matrix((vals,(rows,cols)), shape=(N,theta_dim)).tocsr()

all_theta01 = np.zeros((theta_dim, tot_save))
all_tau01   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

save_idx = 0

for it in tqdm(range(total_iters), desc="BYM p01"):

    phi = X @ curr_theta
    omega = random_polyagamma(1, phi, size=N)

    block_list = []
    for j in range(K):
        if j % 2 == 0:
            block_list.append((1/curr_tau[j]) * prec)
        else:
            block_list.append((1/curr_tau[j]) * diags(np.ones(S)))

    blocks = []
    for i in range(K):
        row = []
        for j in range(K):
            row.append(block_list[i] if i==j else None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    XtOmega = X.T.multiply(omega)
    pos_prec = (XtOmega @ X + curr_prec).tocsc()

    factor = cholesky(pos_prec, mode="simplicial")

    rhs = X.T @ kappa
    mu = factor.solve_A(rhs)

    z = np.random.randn(theta_dim)
    z = z / np.sqrt(factor.D())
    z = factor.solve_Lt(z)
    z = factor.apply_Pt(z)

    curr_theta = mu + z

    for j in range(K):
        sl = slice(j*S,(j+1)*S)
        beta = curr_theta[sl]
        quad = beta @ (prec @ beta) if j%2==0 else beta @ beta
        curr_tau[j] = 1 / np.random.gamma(a_tau+S/2, 1/(b_tau+quad/2))

    if it>=burn and (it-burn)%thin==0:
        all_theta01[:,save_idx] = curr_theta
        all_tau01[:,save_idx] = curr_tau
        save_idx+=1
        if save_idx==tot_save:
            break

np.savez_compressed("bym01_twoComp.npz",
                    all_theta=all_theta01,
                    all_tau=all_tau01)

print("Saved bym01_twoComp.npz")

Using S = 1557


BYM p01: 100%|█████████▉| 5995/6000 [5:22:15<00:16,  3.23s/it]  


Saved bym01_twoComp.npz


In [2]:
# ================================================================
# p10 event: y_t = 1 → y_{t+1} = 0
# ================================================================
loc = np.where(y[:, :-1] == 1)
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:,0], pairs[:,1]))]
pairs[:,1] += 1

row_idx = pairs[:,0]
time_idx = pairs[:,1] - 1
N = len(row_idx)

next_y = y[pairs[:,0], pairs[:,1]]
kappa = (1 - next_y) - 0.5

t = t_scaled[time_idx]

covariates = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*(time_idx+1)/period), np.cos(2*np.pi*(time_idx+1)/period),
    np.sin(2*np.pi*(time_idx+1)/period), np.sin(2*np.pi*(time_idx+1)/period),
    t, t
])

K = covariates.shape[1]
theta_dim = K*S

rows, cols, vals = [], [], []
for i in range(N):
    s = row_idx[i]
    for k in range(K):
        rows.append(i)
        cols.append(s + k*S)
        vals.append(covariates[i,k])

X = coo_matrix((vals,(rows,cols)), shape=(N,theta_dim)).tocsr()

all_theta10 = np.zeros((theta_dim, tot_save))
all_tau10   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

save_idx = 0

for it in tqdm(range(total_iters), desc="BYM p10"):

    phi = X @ curr_theta
    omega = random_polyagamma(1, phi, size=N)

    block_list = []
    for j in range(K):
        if j % 2 == 0:
            block_list.append((1/curr_tau[j]) * prec)
        else:
            block_list.append((1/curr_tau[j]) * diags(np.ones(S)))

    blocks = []
    for i in range(K):
        row = []
        for j in range(K):
            row.append(block_list[i] if i==j else None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    XtOmega = X.T.multiply(omega)
    pos_prec = (XtOmega @ X + curr_prec).tocsc()

    factor = cholesky(pos_prec, mode="simplicial")

    rhs = X.T @ kappa
    mu = factor.solve_A(rhs)

    z = np.random.randn(theta_dim)
    z = z / np.sqrt(factor.D())
    z = factor.solve_Lt(z)
    z = factor.apply_Pt(z)

    curr_theta = mu + z

    for j in range(K):
        sl = slice(j*S,(j+1)*S)
        beta = curr_theta[sl]
        quad = beta @ (prec @ beta) if j%2==0 else beta @ beta
        curr_tau[j] = 1 / np.random.gamma(a_tau+S/2, 1/(b_tau+quad/2))

    if it>=burn and (it-burn)%thin==0:
        all_theta10[:,save_idx] = curr_theta
        all_tau10[:,save_idx] = curr_tau
        save_idx+=1
        if save_idx==tot_save:
            break

np.savez_compressed("bym10_twoComp.npz",
                    all_theta=all_theta10,
                    all_tau=all_tau10)

print("Saved bym10_twoComp.npz")

BYM p10: 100%|█████████▉| 5995/6000 [2:35:10<00:07,  1.55s/it]  


Saved bym10_twoComp.npz


In [4]:
import numpy as np
from scipy.sparse import coo_matrix

# ================================================================
# Rebuild design matrix 
# ================================================================
def build_design_bym(loc_mask, is_p10=False):

    loc = np.where(loc_mask)
    pairs = np.column_stack(loc)
    pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
    pairs[:, 1] += 1

    row_idx  = pairs[:, 0]
    time_idx = pairs[:, 1] - 1
    N = len(row_idx)

    next_y = y[pairs[:, 0], pairs[:, 1]]

    if is_p10:
        y_vec = 1 - next_y
    else:
        y_vec = next_y

    t = t_scaled[time_idx]

    covariates = np.column_stack([
        np.ones(N), np.ones(N),
        np.cos(2*np.pi*(time_idx+1)/period), np.cos(2*np.pi*(time_idx+1)/period),
        np.sin(2*np.pi*(time_idx+1)/period), np.sin(2*np.pi*(time_idx+1)/period),
        t, t
    ])

    K = covariates.shape[1]
    theta_dim = K * S

    rows, cols, vals = [], [], []

    for i in range(N):
        s = row_idx[i]
        for k in range(K):
            rows.append(i)
            cols.append(s + k*S)
            vals.append(covariates[i, k])

    X = coo_matrix((vals, (rows, cols)),
                   shape=(N, theta_dim)).tocsr()

    return X, y_vec


# ================================================================
# Load posterior samples
# ================================================================
bym01 = np.load("bym01_twoComp.npz")
bym10 = np.load("bym10_twoComp.npz")

theta01 = bym01["all_theta"]
theta10 = bym10["all_theta"]

theta01_mean = theta01.mean(axis=1)
theta10_mean = theta10.mean(axis=1)


# ================================================================
# p01 likelihood
# ================================================================
X01, y01 = build_design_bym(
    loc_mask=(y[:, :-1] == 0),
    is_p10=False
)

phi01 = X01 @ theta01_mean

loglik01 = np.sum(
    y01 * phi01 - np.log1p(np.exp(phi01))
)


# ================================================================
# p10 likelihood
# ================================================================
X10, y10 = build_design_bym(
    loc_mask=(y[:, :-1] == 1),
    is_p10=True
)

phi10 = X10 @ theta10_mean

loglik10 = np.sum(
    y10 * phi10 - np.log1p(np.exp(phi10))
)


# ================================================================
# Total
# ================================================================
total_ll = loglik01 + loglik10

print("BYM posterior mean loglik p01 :", loglik01)
print("BYM posterior mean loglik p10 :", loglik10)
print("BYM posterior mean total ll  :", total_ll)

BYM p10 LLH: 100%|██████████| 1000/1000 [00:45<00:00, 21.96it/s]


BYM posterior mean LLH p01: -341078.4495167122
BYM posterior mean LLH p10: -269686.3319628497
TOTAL BYM posterior mean LLH: -610764.7814795619
